# 139 — BoardCache Timing Analysis

Per-operation wall-clock breakdown for `138-BoardCache.py`.

Each cell times one hot-path segment. The summary table at the bottom shows the full budget.

In [1]:
import sys, math, time, timeit
import importlib.util
import polars as pl
from types import SimpleNamespace

# ── Load 138-BoardCache as a module ──────────────────────────────────────────
spec = importlib.util.spec_from_file_location(
    "board_cache",
    r"C:\Users\trant\Documents\Programmation\Orbit Wars\138-BoardCache.py"
)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

Board           = mod.Board
StrategyPipeline = mod.StrategyPipeline
GameConfig      = mod.GameConfig

print("Module loaded OK")

Module loaded OK


In [2]:
# ── Shared test fixture ───────────────────────────────────────────────────────
def make_obs(step=0):
    """Minimal 3-planet obs: planet 0 = fix (owner 0), planet 1 = moving (owner 1), planet 2 = moving (neutral)."""
    planets = [
        [0, 0, 30.0, 50.0, 31.0, 100, 2],
        [1, 1, 70.0, 50.0,  5.0,  20, 2],
        [2, -1, 50.0, 20.0, 4.0,   0, 1],
    ]
    obs = SimpleNamespace(
        planets=[list(p) for p in planets],
        initial_planets=[list(p) for p in planets],
        fleets=[],
        comets=[],
        comet_planet_ids=[],
        angular_velocity=0.01,
        next_fleet_id=0,
        player=0,
        step=step,
    )
    return obs

N_REPS = 200  # repetitions for timeit measurements
results: dict = {}  # timing_label -> ms

obs0 = make_obs(0)
print("Fixture ready.")

Fixture ready.


## 1 — `Board.__init__`

First-call initialisation: nature table, planet_meta, pos window (P×11 rows).

In [3]:
elapsed = timeit.timeit(
    lambda: Board(make_obs(0), step=0, num_agents=2, player_id=0),
    number=N_REPS
)
ms = elapsed / N_REPS * 1000
results["Board.__init__"] = ms
print(f"Board.__init__   {ms:.3f} ms  (mean over {N_REPS} reps)")

Board.__init__   0.391 ms  (mean over 200 reps)


## 2 — `Board.advance`

Slide pos window + sync df_fleet (no new fleets).

In [4]:
board_adv = Board(make_obs(0), step=0, num_agents=2, player_id=0)

def _advance():
    # advance then reset so we can time repeatedly
    board_adv.advance(make_obs(1), step=1)
    board_adv.advance(make_obs(0), step=0)  # back — just to keep window valid

elapsed = timeit.timeit(_advance, number=N_REPS)
ms = elapsed / N_REPS * 1000 / 2  # two advance calls per rep
results["Board.advance"] = ms
print(f"Board.advance    {ms:.3f} ms  (per call, mean over {N_REPS} reps)")

Board.advance    1.990 ms  (per call, mean over 200 reps)


## 3 — `Board.build_base_ships`

Full 11-step production+combat loop → materialise df_planete_ships (P×11 rows).

In [5]:
board_bs = Board(make_obs(0), step=0, num_agents=2, player_id=0)
board_bs.advance(make_obs(0), step=0)

elapsed = timeit.timeit(
    lambda: board_bs.build_base_ships(make_obs(0)),
    number=N_REPS
)
ms = elapsed / N_REPS * 1000
results["Board.build_base_ships"] = ms
print(f"Board.build_base_ships   {ms:.3f} ms  (mean over {N_REPS} reps)")

Board.build_base_ships   0.132 ms  (mean over 200 reps)


## 4 — `Board.build_df_s_slice`

Join ships + pos + nature into df_s, compute planet_disp. Timed at two typical call sites:
- `step_from=0` (current step — used by agent() for the first _02 call)
- `step_from=5` (baseline / per-c0 — used by _04_minimax_search)

In [6]:
board_sl = Board(make_obs(0), step=0, num_agents=2, player_id=0)
board_sl.advance(make_obs(0), step=0)
board_sl.build_base_ships(make_obs(0))

elapsed0 = timeit.timeit(
    lambda: board_sl.build_df_s_slice(board_sl.df_planete_ships, step_from=0),
    number=N_REPS
)
elapsed5 = timeit.timeit(
    lambda: board_sl.build_df_s_slice(board_sl.df_planete_ships, step_from=5),
    number=N_REPS
)
ms0 = elapsed0 / N_REPS * 1000
ms5 = elapsed5 / N_REPS * 1000
results["build_df_s_slice(step=0)"] = ms0
results["build_df_s_slice(step=5)"] = ms5
print(f"build_df_s_slice step_from=0   {ms0:.3f} ms")
print(f"build_df_s_slice step_from=5   {ms5:.3f} ms")

build_df_s_slice step_from=0   5.492 ms
build_df_s_slice step_from=5   8.050 ms


## 5 — `_02_get_all_opportunities`

Polars LazyFrame pipeline — cross-join, swept-pair geometry, angle computation.

In [7]:
board_02 = Board(make_obs(0), step=0, num_agents=2, player_id=0)
board_02.advance(make_obs(0), step=0)
board_02.build_base_ships(make_obs(0))
df_s0, pd0 = board_02.build_df_s_slice(board_02.df_planete_ships, step_from=0)
df_s5, pd5 = board_02.build_df_s_slice(board_02.df_planete_ships, step_from=5)

elapsed0 = timeit.timeit(
    lambda: StrategyPipeline._02_get_all_opportunities(df_s0, pd0, 0).collect(),
    number=N_REPS
)
elapsed5 = timeit.timeit(
    lambda: StrategyPipeline._02_get_all_opportunities(df_s5, pd5, 0).collect(),
    number=N_REPS
)
ms0 = elapsed0 / N_REPS * 1000
ms5 = elapsed5 / N_REPS * 1000
results["_02_get_all_opportunities(step=0)"] = ms0
results["_02_get_all_opportunities(step=5)"] = ms5
print(f"_02 step_from=0   {ms0:.3f} ms")
print(f"_02 step_from=5   {ms5:.3f} ms")

_02 step_from=0   21.805 ms
_02 step_from=5   26.736 ms


## 6 — `_03_filter_collision`

Self-join on angle cones to eliminate blocked lanes.

In [8]:
pa_lf0 = StrategyPipeline._02_get_all_opportunities(df_s0, pd0, 0)
pa_lf5 = StrategyPipeline._02_get_all_opportunities(df_s5, pd5, 0)

elapsed0 = timeit.timeit(
    lambda: StrategyPipeline._03_filter_collision(pa_lf0).collect(),
    number=N_REPS
)
elapsed5 = timeit.timeit(
    lambda: StrategyPipeline._03_filter_collision(pa_lf5).collect(),
    number=N_REPS
)
ms0 = elapsed0 / N_REPS * 1000
ms5 = elapsed5 / N_REPS * 1000
results["_03_filter_collision(step=0)"] = ms0
results["_03_filter_collision(step=5)"] = ms5
print(f"_03 step_from=0   {ms0:.3f} ms")
print(f"_03 step_from=5   {ms5:.3f} ms")

_03 step_from=0   33.908 ms
_03 step_from=5   31.438 ms


## 7 — `Board._recompute_from_sim`

Dirty-row incremental recompute: ≤2 planets × (10 - step_tgt + 1) steps recomputed.

In [9]:
board_rs = Board(make_obs(0), step=0, num_agents=2, player_id=0)
board_rs.advance(make_obs(0), step=0)
board_rs.build_base_ships(make_obs(0))

move = [0, math.atan2(0.0, 40.0), 50]  # planet 0 → planet 1 (eastward)

# Case A: src only dirty (id_tgt=None, so only src planet recomputes)
elapsed_src = timeit.timeit(
    lambda: board_rs._recompute_from_sim(board_rs.df_planete_ships, move, None, None),
    number=N_REPS
)
# Case B: both src and tgt dirty, step_tgt=5 (mid-window)
elapsed_both = timeit.timeit(
    lambda: board_rs._recompute_from_sim(board_rs.df_planete_ships, move, 1, 5),
    number=N_REPS
)
# Case C: step_tgt=1 (nearly immediate, max dirty rows)
elapsed_early = timeit.timeit(
    lambda: board_rs._recompute_from_sim(board_rs.df_planete_ships, move, 1, 2),
    number=N_REPS
)
ms_src   = elapsed_src   / N_REPS * 1000
ms_both  = elapsed_both  / N_REPS * 1000
ms_early = elapsed_early / N_REPS * 1000
results["_recompute_from_sim (src only)"]       = ms_src
results["_recompute_from_sim (src+tgt step=5)"] = ms_both
results["_recompute_from_sim (src+tgt step=2)"] = ms_early
print(f"_recompute src only          {ms_src:.3f} ms")
print(f"_recompute src+tgt step=5    {ms_both:.3f} ms")
print(f"_recompute src+tgt step=2    {ms_early:.3f} ms")

_recompute src only          2.731 ms
_recompute src+tgt step=5    3.291 ms
_recompute src+tgt step=2    1.558 ms


## 8 — `Board.extract_horizon_dict`

Slice df_planete_ships at the last step → dict. Called once per c0 candidate.

In [10]:
board_eh = Board(make_obs(0), step=0, num_agents=2, player_id=0)
board_eh.advance(make_obs(0), step=0)
board_eh.build_base_ships(make_obs(0))

elapsed = timeit.timeit(
    lambda: board_eh.extract_horizon_dict(board_eh.df_planete_ships),
    number=N_REPS
)
ms = elapsed / N_REPS * 1000
results["Board.extract_horizon_dict"] = ms
print(f"extract_horizon_dict   {ms:.3f} ms")

extract_horizon_dict   0.213 ms


## 9 — `Board._apply_sim_fleet` + `Board._evaluate_dict`

Pure-Python hot path: no Polars allocation. Called K times per c0 candidate in the inner loop.

In [11]:
board_ev = Board(make_obs(0), step=0, num_agents=2, player_id=0)
board_ev.advance(make_obs(0), step=0)
board_ev.build_base_ships(make_obs(0))
horizon = board_ev.extract_horizon_dict(board_ev.df_planete_ships)
sim_row = (0, 1, 5, 50)  # (id_src, id_tgt, step_tgt, ships_sent)

elapsed_apply = timeit.timeit(
    lambda: Board._apply_sim_fleet(horizon, sim_row),
    number=N_REPS * 50  # very fast, use more reps
)
elapsed_eval = timeit.timeit(
    lambda: Board._evaluate_dict(horizon, 0),
    number=N_REPS * 50
)
elapsed_combined = timeit.timeit(
    lambda: Board._evaluate_dict(Board._apply_sim_fleet(horizon, sim_row), 0),
    number=N_REPS * 50
)
ms_apply    = elapsed_apply    / (N_REPS * 50) * 1000
ms_eval     = elapsed_eval     / (N_REPS * 50) * 1000
ms_combined = elapsed_combined / (N_REPS * 50) * 1000
results["_apply_sim_fleet"]          = ms_apply
results["_evaluate_dict"]            = ms_eval
results["_apply_sim_fleet+_evaluate"] = ms_combined
print(f"_apply_sim_fleet          {ms_apply*1000:.3f} µs")
print(f"_evaluate_dict            {ms_eval*1000:.3f} µs")
print(f"_apply + _evaluate        {ms_combined*1000:.3f} µs  (hot-path inner loop)")

_apply_sim_fleet          1.096 µs
_evaluate_dict            4.841 µs
_apply + _evaluate        3.238 µs  (hot-path inner loop)


## 10 — `_04_minimax_search` end-to-end

Full two-level minimax: baseline + per-c0 restricted evaluation.
Includes _recompute_from_sim, _02+_03 (changed-ids), and the dict inner loop.

In [12]:
def _run_minimax():
    obs = make_obs(0)
    b = Board(obs, step=0, num_agents=2, player_id=0)
    b.advance(obs, step=0)
    b.build_base_ships(obs)
    df_s, pd_ = b.build_df_s_slice(b.df_planete_ships, step_from=0)
    pa_lf  = StrategyPipeline._02_get_all_opportunities(df_s, pd_, 0)
    safe_lf = StrategyPipeline._03_filter_collision(pa_lf)
    return StrategyPipeline._04_minimax_search(safe_lf, obs, b)

elapsed = timeit.timeit(_run_minimax, number=20)
ms = elapsed / 20 * 1000
results["_04_minimax_search (full)"] = ms
print(f"_04_minimax_search   {ms:.1f} ms  (mean over 20 reps)")

Minimax best move: [0, -0.9854840335446773, 100]  score=(1, 88)
Minimax best move: [0, -0.9854840335446773, 100]  score=(1, 88)
Minimax best move: [0, -0.9854840335446773, 100]  score=(1, 88)
Minimax best move: [0, -0.9854840335446773, 100]  score=(1, 88)
Minimax best move: [0, -0.9854840335446773, 100]  score=(1, 88)
Minimax best move: [0, -0.9854840335446773, 100]  score=(1, 88)
Minimax best move: [0, -0.9854840335446773, 100]  score=(1, 88)
Minimax best move: [0, -0.9854840335446773, 100]  score=(1, 88)
Minimax best move: [0, -0.9854840335446773, 100]  score=(1, 88)
Minimax best move: [0, -0.9854840335446773, 100]  score=(1, 88)
Minimax best move: [0, -0.9854840335446773, 100]  score=(1, 88)
Minimax best move: [0, -0.9854840335446773, 100]  score=(1, 88)
Minimax best move: [0, -0.9854840335446773, 100]  score=(1, 88)
Minimax best move: [0, -0.9854840335446773, 100]  score=(1, 88)
Minimax best move: [0, -0.9854840335446773, 100]  score=(1, 88)
Minimax best move: [0, -0.98548403354467

## 11 — `agent()` full turn

End-to-end: advance + build_base_ships + _02 + _03 + minimax.

In [13]:
# First call (init)
mod.BOARD = None
obs_a = make_obs(0)
obs_a.step = 0
t0 = time.perf_counter()
mod.agent(obs_a)
t1 = time.perf_counter()
ms_init = (t1 - t0) * 1000
results["agent() first call (init+run)"] = ms_init
print(f"agent() first call (init)   {ms_init:.1f} ms")

# Subsequent calls (advance path)
times = []
for step in range(1, 11):
    obs_n = make_obs(step)
    obs_n.step = step
    t0 = time.perf_counter()
    mod.agent(obs_n)
    t1 = time.perf_counter()
    times.append((t1 - t0) * 1000)
ms_avg = sum(times) / len(times)
results["agent() subsequent call (advance+run)"] = ms_avg
print(f"agent() subsequent calls avg {ms_avg:.1f} ms  (steps 1-10)")
print(f"  min={min(times):.1f}  max={max(times):.1f}  all={[f'{t:.0f}' for t in times]}")

Agent called step=0 remainingOverageTime=0
Minimax best move: [0, -0.9854840335446773, 100]  score=(1, 88)
agent() first call (init)   144.4 ms
Agent called step=1 remainingOverageTime=0
Minimax best move: [0, -0.9784010059648592, 100]  score=(1, 88)
Agent called step=2 remainingOverageTime=0
Minimax best move: [0, -0.9713372100969305, 100]  score=(1, 88)
Agent called step=3 remainingOverageTime=0
Minimax best move: [0, -0.9642923005925801, 100]  score=(1, 88)
Agent called step=4 remainingOverageTime=0
Minimax best move: [0, -0.9572659399215045, 100]  score=(1, 88)
Agent called step=5 remainingOverageTime=0
Minimax best move: [0, -0.9502577981522169, 100]  score=(1, 88)
Agent called step=6 remainingOverageTime=0
Minimax best move: [0, -0.9432675527398908, 100]  score=(1, 88)
Agent called step=7 remainingOverageTime=0
Minimax best move: [0, -0.9362948883209862, 100]  score=(1, 88)
Agent called step=8 remainingOverageTime=0
Minimax best move: [0, -0.9293394965144101, 100]  score=(1, 88)


## 12 — Summary table

In [14]:
import polars as pl

rows = [(k, v) for k, v in results.items()]
df_summary = pl.DataFrame(
    {"operation": [r[0] for r in rows], "ms": [r[1] for r in rows]}
).with_columns(
    pl.col("ms").round(4)
).sort("ms", descending=True)

print(df_summary.to_pandas().to_string(index=False))

total_per_turn = results.get("agent() subsequent call (advance+run)", 0)
budget_ms = 1000.0
print(f"\nTurn budget: {budget_ms:.0f} ms")
print(f"Typical turn: {total_per_turn:.1f} ms  ({total_per_turn/budget_ms*100:.1f}% of budget)")

                            operation       ms
            _04_minimax_search (full) 151.3314
agent() subsequent call (advance+run) 144.4988
        agent() first call (init+run) 144.3729
         _03_filter_collision(step=0)  33.9079
         _03_filter_collision(step=5)  31.4381
    _02_get_all_opportunities(step=5)  26.7362
    _02_get_all_opportunities(step=0)  21.8054
             build_df_s_slice(step=5)   8.0497
             build_df_s_slice(step=0)   5.4920
 _recompute_from_sim (src+tgt step=5)   3.2913
       _recompute_from_sim (src only)   2.7313
                        Board.advance   1.9904
 _recompute_from_sim (src+tgt step=2)   1.5577
                       Board.__init__   0.3906
           Board.extract_horizon_dict   0.2134
               Board.build_base_ships   0.1315
                       _evaluate_dict   0.0048
           _apply_sim_fleet+_evaluate   0.0032
                     _apply_sim_fleet   0.0011

Turn budget: 1000 ms
Typical turn: 144.5 ms  (14.4% of budg

## 13 — Breakdown: where does one minimax turn go?

Model the cost of a full `_04_minimax_search` call with N c0 candidates and K c5 candidates.

In [15]:
N = 5   # step-0 candidates (typical: up to 5)
K = 5   # restricted c5 per c0 (typically ≤5)

baseline_02  = results.get("_02_get_all_opportunities(step=5)", 0)
baseline_03  = results.get("_03_filter_collision(step=5)", 0)
recompute    = results.get("_recompute_from_sim (src+tgt step=5)", 0)
horizon      = results.get("Board.extract_horizon_dict", 0)
inner_loop   = results.get("_apply_sim_fleet+_evaluate", 0) * K
build_ships  = results.get("Board.build_base_ships", 0)
build_slice  = results.get("build_df_s_slice(step=5)", 0)

baseline_total    = baseline_02 + baseline_03
per_c0_total      = recompute + build_slice + baseline_02 + baseline_03 + horizon + inner_loop
minimax_total_est = baseline_total + N * per_c0_total

print(f"Modelled minimax cost (N={N}, K={K}):")
print(f"  Baseline _02+_03            : {baseline_total:.2f} ms")
print(f"  Per-c0 _recompute_from_sim  : {recompute:.2f} ms")
print(f"  Per-c0 build_df_s_slice     : {build_slice:.2f} ms")
print(f"  Per-c0 _02+_03              : {baseline_02+baseline_03:.2f} ms")
print(f"  Per-c0 extract_horizon      : {horizon:.2f} ms")
print(f"  Per-c0 inner loop (K={K})    : {inner_loop:.4f} ms")
print(f"  Per-c0 total                : {per_c0_total:.2f} ms")
print(f"  ─────────────────────────────")
print(f"  Estimated minimax total     : {minimax_total_est:.1f} ms")
print(f"  Actual measured             : {results.get('_04_minimax_search (full)', 0):.1f} ms")

Modelled minimax cost (N=5, K=5):
  Baseline _02+_03            : 58.17 ms
  Per-c0 _recompute_from_sim  : 3.29 ms
  Per-c0 build_df_s_slice     : 8.05 ms
  Per-c0 _02+_03              : 58.17 ms
  Per-c0 extract_horizon      : 0.21 ms
  Per-c0 inner loop (K=5)    : 0.0162 ms
  Per-c0 total                : 69.74 ms
  ─────────────────────────────
  Estimated minimax total     : 406.9 ms
  Actual measured             : 151.3 ms
